# Explorando uma API: dados históricos de clima

## Projeto: preparar dados para prever chuva no dia seguinte

Neste notebook, vamos aprender a:

1. entender o que é uma API e consultar sua documentação;
2. instalar e importar dependências;
3. escolher endpoint, parâmetros e variáveis;
4. fazer uma requisição HTTP com `requests`;
5. verificar status, cabeçalhos e formato da resposta;
6. interpretar um JSON e convertê-lo em `DataFrame`;
7. avaliar qualidade e realizar análises iniciais;

**Pergunta do projeto:** usando as condições meteorológicas de hoje, é possível prever se choverá amanhã?

> Este notebook realiza a coleta e a exploração inicial. O treinamento do modelo pode ser desenvolvido em uma etapa posterior.

## 1. Conceitos essenciais

- **API:** interface que permite que programas troquem dados e comandos.
- **Endpoint:** endereço que oferece um recurso específico da API.
- **Requisição:** pedido enviado pelo cliente ao servidor.
- **Resposta:** retorno do servidor, com status, cabeçalhos e corpo.
- **Parâmetros:** informações que detalham o pedido, como cidade, datas e variáveis.
- **JSON:** formato textual estruturado em pares chave–valor e listas.

Neste projeto, usaremos o endpoint:

`https://archive-api.open-meteo.com/v1/archive`

Documentação oficial: https://open-meteo.com/en/docs/historical-weather-api

A Open-Meteo informa que os dados históricos são dados de **reanálise**: observações de várias fontes são combinadas com modelos meteorológicos. Portanto, não são simplesmente medições brutas de uma única estação.

## 2. Dependências

| Biblioteca | Uso no notebook |
|---|---|
| `requests` | enviar a requisição HTTP e receber a resposta |
| `json` | exibir o JSON de maneira organizada |
| `pandas` | organizar e analisar os dados em tabela |
| `matplotlib` | construir gráficos básicos |
| `seaborn` | criar gráficos estatísticos com menos código |

No Google Colab, essas bibliotecas normalmente já estão instaladas. Em um ambiente local, execute a célula abaixo apenas se necessário.

In [6]:
!pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 3. Escolher o nível de detalhe

A API permite solicitar:

- `hourly`: um registro por hora;
- `daily`: variáveis agregadas por dia.

Para o primeiro projeto, vamos usar `daily`. Assim, cada linha do futuro `DataFrame` representará um dia, sem a necessidade de agregar 24 registros horários.

### Variáveis diárias escolhidas

| Variável | Significado | Por que pode ajudar? |
|---|---|---|
| `temperature_2m_max` | temperatura máxima | caracteriza as condições térmicas |
| `temperature_2m_min` | temperatura mínima | permite observar amplitude térmica |
| `temperature_2m_mean` | temperatura média | resume a temperatura diária |
| `relative_humidity_2m_mean` | umidade relativa média | ar úmido favorece formação de nuvens e precipitação |
| `cloud_cover_mean` | cobertura média de nuvens | aproxima a condição de nebulosidade |
| `pressure_msl_mean` | pressão média ao nível do mar | mudanças de pressão podem indicar sistemas meteorológicos |
| `precipitation_sum` | precipitação total do dia | chuva pode persistir de um dia para outro |
| `precipitation_hours` | horas com precipitação | indica duração da chuva atual |
| `wind_speed_10m_max` | maior velocidade do vento | pode acompanhar frentes e mudanças do tempo |
| `shortwave_radiation_sum` | radiação solar diária | tende a diminuir em dias mais nublados |
| `sunshine_duration` | duração do brilho solar | indicador indireto de nebulosidade |

> A existência de relação física não garante que uma variável será útil ao modelo. Isso será avaliado posteriormente com dados e métricas.

In [8]:
variaveis_diarias = [
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "cloud_cover_mean",
    "pressure_msl_mean",
    "precipitation_sum",
    "precipitation_hours",
    "wind_speed_10m_max",
    "shortwave_radiation_sum",
    "sunshine_duration",
]

variaveis_diarias

['temperature_2m_max',
 'temperature_2m_min',
 'temperature_2m_mean',
 'relative_humidity_2m_mean',
 'cloud_cover_mean',
 'pressure_msl_mean',
 'precipitation_sum',
 'precipitation_hours',
 'wind_speed_10m_max',
 'shortwave_radiation_sum',
 'sunshine_duration']

## 4. Definir os parâmetros da requisição

Usaremos as coordenadas aproximadas da cidade de São Paulo. Coordenadas são mais precisas que nomes de cidades e são exigidas por esse endpoint.

### Parâmetros obrigatórios

- `latitude` e `longitude`: localização geográfica;
- `start_date` e `end_date`: intervalo no formato `AAAA-MM-DD`;
- `daily`: variáveis que desejamos obter;
- `timezone`: necessária para variáveis diárias e importante para interpretar corretamente as datas.

O período de 2015 a 2025 gera milhares de dias: é suficiente para uma primeira experiência, mas ainda leve para um notebook didático.

In [9]:
url = "https://archive-api.open-meteo.com/v1/archive"

parametros = {
    "latitude": -23.55,
    "longitude": -46.63,
    "start_date": "2015-01-01",
    "end_date": "2025-12-31",
    "daily": variaveis_diarias,  #parametros definidos anteriormente
    "timezone": "America/Sao_Paulo",
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm",
}

parametros

{'latitude': -23.55,
 'longitude': -46.63,
 'start_date': '2015-01-01',
 'end_date': '2025-12-31',
 'daily': ['temperature_2m_max',
  'temperature_2m_min',
  'temperature_2m_mean',
  'relative_humidity_2m_mean',
  'cloud_cover_mean',
  'pressure_msl_mean',
  'precipitation_sum',
  'precipitation_hours',
  'wind_speed_10m_max',
  'shortwave_radiation_sum',
  'sunshine_duration'],
 'timezone': 'America/Sao_Paulo',
 'temperature_unit': 'celsius',
 'wind_speed_unit': 'kmh',
 'precipitation_unit': 'mm'}

### Atividade rápida

Antes de executar a requisição, responda:

1. Qual é o endpoint?

```
https://archive-api.open-meteo.com/v1/archive
```
2. Quais parâmetros identificam o lugar?
```
latitude e longitude
```
3. Quais parâmetros definem o período?
```
start_date e end_date
```
4. Por que foi escolhido `daily`, e não `hourly`?
```
para ter os dados agregados por dia e não hora
```
5. Quantos dias você espera receber aproximadamente?
```
4.019
```

## 5. Requisição HTTP

`requests.get()` envia uma requisição do tipo **GET**, usada para consultar um recurso.

- `params=parametros` monta a *query string* corretamente;
- `timeout=30` impede que o programa espere indefinidamente;
- o resultado é um objeto `Response`, ainda não um dicionário Python.

In [10]:
try:
    resposta = requests.get(
        url,
        params=parametros,
        timeout=30)
except requests.exceptions.RequestException as erro:
    raise RuntimeError(f"Não foi possível acessar a API: {erro}") from erro

print("Tipo do objeto:", type(resposta))
print("URL montada:", resposta.url)
print("Status HTTP:", resposta.status_code)
print("Content-Type:", resposta.headers.get("Content-Type"))

Tipo do objeto: <class 'requests.models.Response'>
URL montada: https://archive-api.open-meteo.com/v1/archive?latitude=-23.55&longitude=-46.63&start_date=2015-01-01&end_date=2025-12-31&daily=temperature_2m_max&daily=temperature_2m_min&daily=temperature_2m_mean&daily=relative_humidity_2m_mean&daily=cloud_cover_mean&daily=pressure_msl_mean&daily=precipitation_sum&daily=precipitation_hours&daily=wind_speed_10m_max&daily=shortwave_radiation_sum&daily=sunshine_duration&timezone=America%2FSao_Paulo&temperature_unit=celsius&wind_speed_unit=kmh&precipitation_unit=mm
Status HTTP: 200
Content-Type: application/json; charset=utf-8


## 6. Interpretar a resposta

Alguns status HTTP frequentes:

| Status | Significado |
|---:|---|
| 200 | requisição concluída com sucesso |
| 400 | parâmetros ausentes ou incorretos |
| 404 | recurso não encontrado |
| 429 | limite de requisições excedido |
| 500 | erro interno do servidor |

`raise_for_status()` interrompe a execução se o status indicar erro. Antes de chamar `.json()`, também verificaremos se o servidor declarou conteúdo JSON.

In [11]:
resposta.raise_for_status()

content_type = resposta.headers.get("Content-Type", "").lower()
if "json" not in content_type:
    raise ValueError(
        "A API não retornou JSON. "
        f"Content-Type recebido: {content_type or 'não informado'}"
    )

dados = resposta.json()

print("Tipo de resposta:", type(resposta))
print("Tipo após .json():", type(dados))
print("Chaves principais:", list(dados.keys()))

Tipo de resposta: <class 'requests.models.Response'>
Tipo após .json(): <class 'dict'>
Chaves principais: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']


In [12]:
# Visualização parcial e organizada do documento JSON.
# Limitei a saída para não ocupar muitas páginas do notebook.
texto_json = json.dumps(dados, indent=2, ensure_ascii=False)
print(texto_json[:1000])

{
  "latitude": -23.514938,
  "longitude": -46.610504,
  "generationtime_ms": 2951.0034322738647,
  "utc_offset_seconds": -10800,
  "timezone": "America/Sao_Paulo",
  "timezone_abbreviation": "GMT-3",
  "elevation": 737.0,
  "daily_units": {
    "time": "iso8601",
    "temperature_2m_max": "°C",
    "temperature_2m_min": "°C",
    "temperature_2m_mean": "°C",
    "relative_humidity_2m_mean": "%",
    "cloud_cover_mean": "%",
    "pressure_msl_mean": "hPa",
    "precipitation_sum": "mm",
    "precipitation_hours": "h",
    "wind_speed_10m_max": "km/h",
    "shortwave_radiation_sum": "MJ/m²",
    "sunshine_duration": "s"
  },
  "daily": {
    "time": [
      "2015-01-01",
      "2015-01-02",
      "2015-01-03",
      "2015-01-04",
      "2015-01-05",
      "2015-01-06",
      "2015-01-07",
      "2015-01-08",
      "2015-01-09",
      "2015-01-10",
      "2015-01-11",
      "2015-01-12",
      "2015-01-13",
      "2015-01-14",
      "2015-01-15",
      "2015-01-16",
      "2015-01-17",
 

## 7. Entender a estrutura recebida

O JSON tem informações gerais no primeiro nível. Os dados tabulares estão em `daily`, enquanto `daily_units` informa as unidades.

Observe uma característica importante: dentro de `daily`, cada chave contém uma lista. Os elementos na mesma posição pertencem ao mesmo dia. É essa estrutura que o Pandas converterá em linhas e colunas.

In [13]:
print("Metadados:")
print("Coordenada retornada:", dados["latitude"], dados["longitude"])
print("Fuso horário:", dados["timezone"])
print("Elevação:", dados["elevation"], "m")

print("\nUnidades das variáveis:")
display(pd.Series(dados["daily_units"], name="unidade").to_frame())

Metadados:
Coordenada retornada: -23.514938 -46.610504
Fuso horário: America/Sao_Paulo
Elevação: 737.0 m

Unidades das variáveis:


,unidade
time,iso8601
temperature_2m_max,°C
temperature_2m_min,°C
temperature_2m_mean,°C
relative_humidity_2m_mean,%
cloud_cover_mean,%
pressure_msl_mean,hPa
precipitation_sum,mm
precipitation_hours,h
wind_speed_10m_max,km/h


## 8. Converter o JSON para DataFrame

Um `DataFrame` organiza os dados em linhas e colunas e facilita filtros, cálculos, gráficos e preparação para aprendizado de máquina.

In [14]:
df = pd.DataFrame(dados["daily"])
df["time"] = pd.to_datetime(df["time"])

df.head()

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,relative_humidity_2m_mean,cloud_cover_mean,pressure_msl_mean,precipitation_sum,precipitation_hours,wind_speed_10m_max,shortwave_radiation_sum,sunshine_duration
0,2015-01-01,31.5,21.8,26.3,72,74,1012.4,0.3,1.0,19.9,29.60,47560.45
1,2015-01-02,30.9,22.2,25.9,75,65,1012.8,7.8,6.0,20.9,26.71,47593.65
2,2015-01-03,25.0,21.4,22.8,92,93,1015.6,5.7,11.0,16.3,18.41,34175.66
3,2015-01-04,25.6,21.2,22.7,89,95,1016.7,20.9,12.0,14.6,16.83,28601.84
4,2015-01-05,25.1,20.3,22.1,90,92,1017.1,5.9,7.0,14.6,13.23,26341.95


In [15]:
print(dados.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


In [16]:
print("Dimensão (linhas, colunas):", df.shape)
print("Período:", df["time"].min().date(), "até", df["time"].max().date())
print("\nTipos de dados:")
display(df.dtypes.to_frame("tipo"))

Dimensão (linhas, colunas): (4018, 12)
Período: 2015-01-01 até 2025-12-31

Tipos de dados:


,tipo
time,datetime64[us]
temperature_2m_max,float64
temperature_2m_min,float64
temperature_2m_mean,float64
relative_humidity_2m_mean,int64
cloud_cover_mean,int64
pressure_msl_mean,float64
precipitation_sum,float64
precipitation_hours,float64
wind_speed_10m_max,float64


## 9. Auditoria inicial da qualidade

---



Antes de analisar ou treinar um modelo, devemos verificar:

- datas repetidas;
- datas ausentes no intervalo;
- valores ausentes (`NaN`);
- linhas duplicadas
- verificar a dimensão do DataFrame
- verificar as primeiras linhas
- verificar as ultimas linhas
- verificar nome das colunas
- usar info()
- usar describe()
- visualizar os dados de 1 coluna
- visualuzar os dados de 2 colunas
- usar o loc e o iloc
- Quantos dias a precipitação foi 0


In [22]:
df["time"].duplicated().sum()

np.int64(0)

In [20]:
df.isna().sum()

time                         0
temperature_2m_max           0
temperature_2m_min           0
temperature_2m_mean          0
relative_humidity_2m_mean    0
cloud_cover_mean             0
pressure_msl_mean            0
precipitation_sum            0
precipitation_hours          0
wind_speed_10m_max           0
shortwave_radiation_sum      0
sunshine_duration            0
dtype: int64

In [21]:
df.duplicated().sum()

np.int64(0)

In [19]:
print(f"volume dos dados e colunas: {df.shape}")

volume dos dados e colunas: (4018, 12)


In [23]:
df.head()

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,relative_humidity_2m_mean,cloud_cover_mean,pressure_msl_mean,precipitation_sum,precipitation_hours,wind_speed_10m_max,shortwave_radiation_sum,sunshine_duration
0,2015-01-01,31.5,21.8,26.3,72,74,1012.4,0.3,1.0,19.9,29.60,47560.45
1,2015-01-02,30.9,22.2,25.9,75,65,1012.8,7.8,6.0,20.9,26.71,47593.65
2,2015-01-03,25.0,21.4,22.8,92,93,1015.6,5.7,11.0,16.3,18.41,34175.66
3,2015-01-04,25.6,21.2,22.7,89,95,1016.7,20.9,12.0,14.6,16.83,28601.84
4,2015-01-05,25.1,20.3,22.1,90,92,1017.1,5.9,7.0,14.6,13.23,26341.95


In [24]:
df.tail()

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,relative_humidity_2m_mean,cloud_cover_mean,pressure_msl_mean,precipitation_sum,precipitation_hours,wind_speed_10m_max,shortwave_radiation_sum,sunshine_duration
4013,2025-12-27,35.1,22.7,27.1,66,51,1013.6,4.4,2.0,9.7,26.75,47943.35
4014,2025-12-28,35.0,23.2,27.9,61,48,1012.0,13.2,6.0,12.6,27.64,48255.21
4015,2025-12-29,33.3,22.6,26.2,71,69,1012.0,5.0,7.0,13.6,27.59,46254.16
4016,2025-12-30,31.3,21.0,24.9,78,71,1013.0,8.0,7.0,12.2,27.12,46018.42
4017,2025-12-31,29.5,20.2,23.9,81,80,1013.9,8.0,10.0,14.9,22.06,37814.08


In [25]:
df.columns

Index(['time', 'temperature_2m_max', 'temperature_2m_min',
       'temperature_2m_mean', 'relative_humidity_2m_mean', 'cloud_cover_mean',
       'pressure_msl_mean', 'precipitation_sum', 'precipitation_hours',
       'wind_speed_10m_max', 'shortwave_radiation_sum', 'sunshine_duration'],
      dtype='str')

In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4018 entries, 0 to 4017
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   time                       4018 non-null   datetime64[us]
 1   temperature_2m_max         4018 non-null   float64       
 2   temperature_2m_min         4018 non-null   float64       
 3   temperature_2m_mean        4018 non-null   float64       
 4   relative_humidity_2m_mean  4018 non-null   int64         
 5   cloud_cover_mean           4018 non-null   int64         
 6   pressure_msl_mean          4018 non-null   float64       
 7   precipitation_sum          4018 non-null   float64       
 8   precipitation_hours        4018 non-null   float64       
 9   wind_speed_10m_max         4018 non-null   float64       
 10  shortwave_radiation_sum    4018 non-null   float64       
 11  sunshine_duration          4018 non-null   float64       
dtypes: datetime64[us]

In [30]:
df_describe = df[['temperature_2m_max', 'temperature_2m_min',
       'temperature_2m_mean', 'relative_humidity_2m_mean', 'cloud_cover_mean',
       'pressure_msl_mean', 'precipitation_sum', 'precipitation_hours',
       'wind_speed_10m_max', 'shortwave_radiation_sum', 'sunshine_duration']]

df_describe.describe()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,relative_humidity_2m_mean,cloud_cover_mean,pressure_msl_mean,precipitation_sum,precipitation_hours,wind_speed_10m_max,shortwave_radiation_sum,sunshine_duration
count,4018.000000,4018.000000,4018.000000,4018.000000,4018.000000,4018.000000,4018.000000,4018.000000,4018.000000,4018.000000,4018.000000
mean,24.932852,15.713290,19.571379,80.221503,66.007964,1016.936262,3.438875,4.688153,17.647287,17.354582,32272.882267
std,4.067217,3.315537,3.309032,9.687202,27.801618,4.135935,6.454437,5.685743,4.853329,5.986767,12181.238364
min,11.800000,1.400000,6.200000,28.000000,0.000000,1003.000000,0.000000,0.000000,4.700000,1.350000,0.000000
25%,22.300000,13.400000,17.300000,76.000000,47.000000,1014.000000,0.000000,0.000000,14.200000,13.640000,27985.042500
50%,25.200000,16.000000,19.800000,83.000000,71.000000,1016.600000,0.500000,3.000000,17.400000,17.155000,36342.225000
75%,27.700000,18.300000,22.000000,87.000000,90.000000,1019.600000,3.900000,8.000000,20.800000,21.727500,40482.880000
max,38.100000,23.400000,28.700000,98.000000,100.000000,1030.800000,67.700000,24.000000,43.100000,31.710000,48434.030000


In [33]:
df[["temperature_2m_max"]].head()

,temperature_2m_max
0,31.5
1,30.9
2,25.0
3,25.6
4,25.1


In [34]:
df[["temperature_2m_max", "relative_humidity_2m_mean"]].head()

,temperature_2m_max,relative_humidity_2m_mean
0,31.5,72
1,30.9,75
2,25.0,92
3,25.6,89
4,25.1,90


In [35]:
df.loc[df["time"] == "2015-01-01"]

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,relative_humidity_2m_mean,cloud_cover_mean,pressure_msl_mean,precipitation_sum,precipitation_hours,wind_speed_10m_max,shortwave_radiation_sum,sunshine_duration
0,2015-01-01,31.5,21.8,26.3,72,74,1012.4,0.3,1.0,19.9,29.6,47560.45


In [37]:
df.iloc[:5, :3]

,time,temperature_2m_max,temperature_2m_min
0,2015-01-01,31.5,21.8
1,2015-01-02,30.9,22.2
2,2015-01-03,25.0,21.4
3,2015-01-04,25.6,21.2
4,2015-01-05,25.1,20.3


In [41]:
dias_precipitacao = (df["precipitation_sum"] == 0).sum()
dias_precipitacao

np.int64(1509)

In [43]:
df.to_csv('input_modelo.csv', index=False)

## Dicionário de dados

| Coluna | Tipo | Unidade | Descrição | Origem |
|---|---|---|---|---|
| `time` | data (`datetime64`) | AAAA-MM-DD | Data de referência do registro diário | API (bruto) |
| `temperature_2m_max` | float | °C | Temperatura máxima do dia, medida a 2 m de altura | API (bruto) |
| `temperature_2m_min` | float | °C | Temperatura mínima do dia | API (bruto) |
| `temperature_2m_mean` | float | °C | Temperatura média do dia | API (bruto) |
| `relative_humidity_2m_mean` | int | % | Umidade relativa média do ar no dia | API (bruto) |
| `cloud_cover_mean` | int | % | Cobertura média de nuvens no dia | API (bruto) |
| `pressure_msl_mean` | float | hPa | Pressão atmosférica média ao nível do mar | API (bruto) |
| `precipitation_sum` | float | mm | Total de precipitação (chuva) acumulada no dia | API (bruto) |
| `precipitation_hours` | float | h | Número de horas do dia com precipitação | API (bruto) |
| `wind_speed_10m_max` | float | km/h | Velocidade máxima do vento a 10 m de altura | API (bruto) |
| `shortwave_radiation_sum` | float | MJ/m² | Radiação solar total recebida no dia | API (bruto) |
| `sunshine_duration` | float | s | Duração total de insolação (sol) no dia | API (bruto) |
| `choveu_hoje` | int (0/1) | binário | 1 se `precipitation_sum` do próprio dia ultrapassou o limiar definido, 0 caso contrário | Derivada |
| `choveu_amanha` | int (0/1) | binário — **alvo** | 1 se choveu no dia seguinte, 0 caso contrário. É o que o modelo vai prever | Derivada (target) |

In [ ]:
print("Período:", df["time"].min().date(), "até", df["time"].max().date())